In [173]:
from dotenv import load_dotenv
load_dotenv()

True

In [174]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore

from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field


In [175]:
#Document Load
loaders= PyPDFLoader("../data/Vineet.pdf")
docs = loaders.load()

##spliter
splitter = RecursiveCharacterTextSplitter(chunk_size=200,chunk_overlap=40)
doc = splitter.split_documents(docs)

#embedding and Vector db

embedding = MistralAIEmbeddings(model="mistral-embed-2312")

vector_store = InMemoryVectorStore.from_documents(
    documents = doc,
    embedding= embedding
)




In [176]:
llm = ChatGroq(model="openai/gpt-oss-20b")

In [177]:
class RagState(BaseModel):
    question:str = Field(description="User will ask the query")
    document: list = []
    context: str =  Field(description="Context data from user question", default="")
    answer: str = Field(description="Final Answer", default="")


In [178]:
#question -> retrieve -> context -> generate -> end


def retrieve_node(state:RagState) -> RagState:
    docs = vector_store.similarity_search(query=state.question)
    state.document = docs
    return state

In [179]:
def create_context_node(state:RagState) -> RagState:
    context = ""
    for doc in state.document:
        context += doc.page_content + "\n\n"
    state.context = context
    return state

In [180]:
def generate_node(state:RagState)->RagState:
    prompt=f"""
        You are a assistant and provide the answer for the user question based on the provided content. If you don't know the relavent answer  then just say 'I don't know this shit'

        Agrs:
            Context is: {state.context},
            Question is: {state.question}
    """
    res = llm.invoke(prompt)
    state.answer = res.content
    return state

In [181]:
graph = StateGraph(RagState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("context",create_context_node)
graph.add_node("generate", generate_node)


graph.add_edge(START, "retrieve")
graph.add_edge("retrieve","context")
graph.add_edge("context", "generate")
graph.add_edge("generate", END)

graph = graph.compile()

In [186]:
res = graph.invoke({"question":"Share me a details of Hospital"})
print(res["answer"])

**Motherland Hospital – Centre 5054**

| Item | Detail |
|------|--------|
| **Name** | Motherland Hospital |
| **Centre Code** | 5054 |
| **Address** | NH‑01, Sector 119 |
| **Phone / Contact** | 9953777444 |
| **Location** | (Sector 119 – specific city/state not provided in the excerpt) |
| **Services Mentioned** | Hematology lab services (sample collection and reporting) |

*Note: The information above is taken directly from the provided context. No additional public details about the hospital (e.g., specialties, accreditation, or capacity) are available in the excerpt.*
